In [12]:
using DelimitedFiles, Statistics, Printf, CSV, DataFrames, LinearAlgebra;
DIR_SOURCE_CODD = "../src"
include("$(DIR_SOURCE_CODD)/tools_imputation_minimum.jl");

In [4]:
DIR_DATA = "../raw-data"
assay_csv = CSV.read( "$(DIR_DATA)/catnap_single.csv", DataFrame )
env_feature_csv = CSV.read("$(DIR_DATA)/env_feature_catnap.txt", DataFrame );

In [5]:
## Adding numerical 
DIR_PROCESSED = "../processed-data"
(msa_Gly, seq_headder_Gly, seq_len_Gly) = read_fastafile("$(DIR_DATA)/virseqs_aa_O_Dec2025.fasta"); # len 1133; N=2716
fname_features = "$(DIR_PROCESSED)/var_reg_char_25-12-30_14-8_V1toV5_merged.csv"; # N=2716 the name exactly matchs up with seq_headder_Gly
csv_features_raw = CSV.read(fname_features, DataFrame); 

### Creating the effective low-dim sequences 

In [6]:
M_min = length(unique(assay_csv.Virus))
virus_unique_catnap = unique(assay_csv.Virus)[1:(M_min)] # This is a temporal. 
idx_look_csv = [x ∈ virus_unique_catnap for x in assay_csv.Virus];
assay_csv = copy(assay_csv[idx_look_csv, :])
#------------------------------------------------------------------------#
antibody_unique_catnap = unique(assay_csv.Antibody);
accession_in_msa = []
unique_key_in_msa = []
for x in seq_headder_Gly
    x_splited = split(x, ".")
    push!(accession_in_msa, x_splited[end])
    push!(unique_key_in_msa, x_splited[end-1])
end;

In [7]:
antibody_unique_catnap_single_abs = []
antibody_unique_catnap_multiple_abs = []
antibody_unique_catnap_polyclonal_abs = []
for x in antibody_unique_catnap
    x_splited = split(x, "polyclonal")
    if(length(x_splited)==1)
        x_splited2 = split(x, ['/', '+'])
        if(length(x_splited2)==1)
            push!(antibody_unique_catnap_single_abs, x)
        else
            push!(antibody_unique_catnap_multiple_abs, x)
        end
    else
        push!(antibody_unique_catnap_polyclonal_abs, x)
    end
end

In [15]:
map_virus2num = Dict(zip(virus_unique_catnap, collect(1:length(virus_unique_catnap))));
maping_unique_key_accession = Dict(zip(unique_key_in_msa, accession_in_msa));
(IC50_single, IC80_single, HillCoeff_mat, IIP_mat, IC50_single_raw, IC80_single_raw) = get_IC50_IC80_Hill_IIP(assay_csv, virus_unique_catnap, antibody_unique_catnap_single_abs, map_virus2num);


In [16]:
seq_ID_from_msa = []

for x in seq_headder_Gly
    x_split = split(x, ".")
    push!(seq_ID_from_msa, x_split[end-1]) # I know the this position gives the accession ID. 
end;

idx_HXB2 = [!isnothing(match(r"HXB2", string(x))) for x in seq_headder_Gly];
i_HXB2 = findall(idx_HXB2)[1];
virus_names_common_in_msa_catnap = intersect(virus_unique_catnap, seq_ID_from_msa[.!idx_HXB2]);
idx_look_vrs = [x ∈ virus_names_common_in_msa_catnap for x in virus_unique_catnap ];

IC50_single_w_seq = copy(IC50_single[:, idx_look_vrs]);
IC80_single_w_seq = copy(IC80_single[:, idx_look_vrs]);
HillCoeff_w_seq = copy(HillCoeff_mat[:, idx_look_vrs]);
IIP_w_seq = copy(IIP_mat[:, idx_look_vrs]);

In [17]:
# Set of virus names in the IC50 after filtering. 
virus_unique_catnap_filtered = copy(virus_unique_catnap[ [x ∈ virus_names_common_in_msa_catnap for x in virus_unique_catnap] ]);
# Set mapping the virus name to the number
map_filtered_virus2num = Dict(zip(virus_unique_catnap_filtered, collect(1:length(virus_unique_catnap_filtered))));

In [18]:
# selecting sequences that are selected. 
idx_look_vrs_in_msa = [x ∈ virus_names_common_in_msa_catnap for x in seq_ID_from_msa];
idx_look_vrs_in_msa[i_HXB2] = true

# making msa and seq_header that are shared with CATNAP. Note that these are not yet sorted. 
msa_filtered = copy(msa_Gly[idx_look_vrs_in_msa])
seq_headder_filtered = copy(seq_headder_Gly[idx_look_vrs_in_msa]);
idx_HXB2_filtered = [!isnothing(match(r"HXB2", x)) for x in seq_headder_filtered];
i_HXB2_filtered = findall(idx_HXB2_filtered)[1];

# taking only sequence name as we use them for the key of the mapping. 
virus_names_in_filtered_msa = [split(x, ".")[end-1] for x in seq_headder_filtered[.!idx_HXB2_filtered]];

# Now, I want to set the index by giving the sequence id of msa, and map_filtered_virus2num
idx_sort_msa = []
for x in virus_unique_catnap_filtered
    i = collect(1:length(virus_names_in_filtered_msa))[virus_names_in_filtered_msa .== x]
    push!(idx_sort_msa, i[1])
end;

@assert virus_names_in_filtered_msa[idx_sort_msa] == virus_unique_catnap_filtered;

# now i sort the seqeucnes of canap and msa. 
msa_filtered_sorted = copy(msa_filtered[.!idx_HXB2_filtered][idx_sort_msa]);
seq_headder_filtered_sorted = copy(seq_headder_filtered[.!idx_HXB2_filtered][idx_sort_msa]);

In [21]:
# Taking only none-gap sites, note sequences are aligned with HXB2 sequences. 
idx_nongap = msa_filtered[i_HXB2_filtered] .!= "-" # should be different from msa_filtered_sorted, which does not cotan HXB2. 
q, L = length(Alpha_set_O), count(idx_nongap)
M =  size(msa_filtered_sorted,1)
msa_act = zeros(Int, M, L)
for n in 1:M
    seq_cut = msa_filtered_sorted[n][idx_nongap]
    idx = seq_cut .== "#" # consider it as alignment gap
    if(count(idx)>0) seq_cut[idx] .= "-" end
    msa_act[n,:] = [aa2num_O[x] for x in seq_cut]
end;

In [25]:
# --- since i don't export the projected seq and PCA spaces, some of the lines here and maybe above could be removed. 
msa_one_hot = one_hot_encode(msa_act, q, M, L);
idx_ply_oh = [length(unique(msa_one_hot[:, i]))>1 for i in 1:length(msa_one_hot[1,:]) ];
msa_one_hot_ply = copy(msa_one_hot[:, idx_ply_oh]);

mean_onehot_poly = [mean(msa_one_hot_ply[:, i]) for i in 1:length(msa_one_hot_ply[1, :])];
scale_w = 1.0 / (size(msa_one_hot_ply,1) - 1)
C_one_hot = (scale_w * msa_one_hot_ply)' * msa_one_hot_ply - mean_onehot_poly * mean_onehot_poly';
@time (evl, evt) = eigen(C_one_hot);

#rank_PCA = 100;# effective sequence length after the projection. This can be a bit longer in real. 
rank_PCA = count(evl .> 1e-5)
PCA_space = zeros(size(evt,1), rank_PCA)
for k in 1:rank_PCA
    PCA_space[:, k] = copy(evt[:, end+1-k])
end
projected_seq_PCA = PCA_space' * msa_one_hot_ply';
subtract_projected = mean(projected_seq_PCA)

# probably, the normalization should be done for each row or without normalization. 
std_projected = std(projected_seq_PCA);
projected_seq_PCA = (1.0/std_projected) * (projected_seq_PCA .- subtract_projected);

# The first element is always one, which corresponds to the cross-section term.
projected_seq_PCA_copy = ones(size(projected_seq_PCA,1)+1, size(projected_seq_PCA,2))
projected_seq_PCA_copy[2:end, :] = copy(projected_seq_PCA);

143.944038 seconds (1.72 M allocations: 1.621 GiB, 0.51% compilation time)


In [26]:
DIR_OUT = "../output"
writedlm("$(DIR_OUT)/IC50_single_w_seq_O.txt", IC50_single_w_seq)
writedlm("$(DIR_OUT)/IC80_single_w_seq_O.txt", IC80_single_w_seq)
# Inf --> NaN
HillCoeff_w_seq[isinf.(HillCoeff_w_seq)] .= NaN
IIP_w_seq[isinf.(IIP_w_seq)] .= NaN
writedlm("$(DIR_OUT)/HillCoeff_w_seq_O.txt", HillCoeff_w_seq)
writedlm("$(DIR_OUT)/IIP_w_seq_O.txt", IIP_w_seq)

writedlm("$(DIR_OUT)/antibody_unique_catnap_single_abs_O.txt",  antibody_unique_catnap_single_abs)
writedlm("$(DIR_OUT)/virus_unique_catnap_filtered_O.txt",  virus_unique_catnap_filtered)
writedlm("$(DIR_OUT)/msa_act_O.txt",  msa_act)
writedlm("$(DIR_OUT)/C_one_hot_O.txt",  C_one_hot)
writedlm("$(DIR_OUT)/msa_filtered_O.txt",  msa_filtered);
writedlm("$(DIR_OUT)/seq_headder_filtered_O.txt",  seq_headder_filtered);
writedlm("$(DIR_OUT)/evl_O.txt",  evl)
writedlm("$(DIR_OUT)/evt_O.txt",  evt);
writedlm("$(DIR_OUT)/idx_ply_oh.txt",  idx_ply_oh);
writedlm("$(DIR_OUT)/projected_seq_PCA_O.txt",  projected_seq_PCA_copy);


##  Processing the sequence features 

In [27]:
virsname2order_features = Dict(zip(string.(virus_unique_catnap_filtered), collect(1:length(virus_unique_catnap_filtered))));
#csv_features_raw
seq_name_in_features  = []
for x in csv_features_raw.Name
    push!(seq_name_in_features, split(x, ".")[end-1] )
end;

N_F, L_F = length(virus_unique_catnap_filtered), length(csv_features_raw[1, 2:end])
mat_features_raw = zeros(N_F, L_F)
for n in 1:length(seq_name_in_features)
    name_n = seq_name_in_features[n]
    if(name_n ∈ virus_unique_catnap_filtered)
        n_eff = virsname2order_features[name_n]
        mat_features_raw[n_eff, :] = copy([csv_features_raw[n, i+1] for i in 1:L_F])
    end
end;

In [28]:
mat_features_raw
mean_F = mat_features_raw' * ( (1.0/N_F) * ones(N_F) )
Cov_F = mat_features_raw' * ( (1.0/N_F) * mat_features_raw) - mean_F * mean_F';
(evl_F, evt_F) = eigen(Cov_F);
evl_F, evt_F = real.(evl_F), real.(evt_F)

PCA_space_F = zeros(size(evt_F))
for k in 1:size(evt_F,2)
    PCA_space_F[:, k] = copy(evt_F[:, end+1-k])
end

projected_seq_PCA_F = PCA_space_F' * mat_features_raw'

subtract_projected_F = mean(projected_seq_PCA_F)
std_projected_F = std(projected_seq_PCA_F);
projected_seq_PCA_F = (1.0/std_projected_F) * (projected_seq_PCA_F .- subtract_projected_F);

projected_seq_PCA_copy_F = ones(size(projected_seq_PCA_F,1)+1, size(projected_seq_PCA_F,2))
projected_seq_PCA_copy_F[2:end, :] = copy(projected_seq_PCA_F);

In [29]:
# NOTE: I didn't make mean and std for projected values
writedlm("$(DIR_OUT)/Cov_F_O.txt",  Cov_F)
writedlm("$(DIR_OUT)/mat_features.txt", mat_features_raw)
writedlm("$(DIR_OUT)/evl_O_w_features.txt", evl_F)
writedlm("$(DIR_OUT)/evt_O_w_features.txt", evt_F);
writedlm("$(DIR_OUT)/projected_seq_PCA_O_w_features.txt", projected_seq_PCA_F); # this not include 1 in this case.
